In [1]:
!pip -q install "mcp>=2,<3" groq

In [2]:
from google.colab import userdata
from groq import Groq

GROQ_API_KEY = userdata.get("GROQ_API_KEY")
if not GROQ_API_KEY:
    raise ValueError("Add GROQ_API_KEY in Colab Secrets.")

groq_client = Groq(api_key=GROQ_API_KEY)
MODEL_NAME = "openai/gpt-oss-120b"
print("Groq connected successfully!")

Groq connected successfully!


In [3]:
%%writefile nutrition_mcp_server.py
from mcp.server.mcpserver import MCPServer

# Demonstration data: values per 100 g.
# 165 kcal is the value specified in the assignment.
# Protein is an illustrative sample value; replace with your dataset if needed.
FOOD_DATA = {
    "chicken breast": {
        "calories_per_100g": 165,
        "protein_per_100g": 31.0
    }
}

mcp = MCPServer("Nutrition MCP Server")


@mcp.tool()
def get_calories(food: str, quantity_g: float = 100.0) -> dict:
    """Return estimated calories (kcal) for a food and quantity in grams."""
    key = food.lower().strip()
    if quantity_g <= 0:
        return {"error": "quantity_g must be greater than zero."}
    if key not in FOOD_DATA:
        return {"error": f"Food '{food}' is not in the sample dataset.",
                "available_foods": list(FOOD_DATA)}
    value = FOOD_DATA[key]["calories_per_100g"] * quantity_g / 100
    return {"food": key, "quantity_g": quantity_g,
            "calories_kcal": round(value, 1)}


@mcp.tool()
def get_protein(food: str, quantity_g: float = 100.0) -> dict:
    """Return estimated protein in grams for a food and quantity in grams."""
    key = food.lower().strip()
    if quantity_g <= 0:
        return {"error": "quantity_g must be greater than zero."}
    if key not in FOOD_DATA:
        return {"error": f"Food '{food}' is not in the sample dataset.",
                "available_foods": list(FOOD_DATA)}
    value = FOOD_DATA[key]["protein_per_100g"] * quantity_g / 100
    return {"food": key, "quantity_g": quantity_g,
            "protein_g": round(value, 1)}


if __name__ == "__main__":
    print("Starting Nutrition MCP Server at http://127.0.0.1:8000/mcp")
    mcp.run(transport="streamable-http", host="127.0.0.1", port=8000)


Writing nutrition_mcp_server.py


In [4]:
import subprocess, sys, time, socket

server_log = open("/content/nutrition_mcp_server.log", "w")
server_process = subprocess.Popen(
    [sys.executable, "nutrition_mcp_server.py"],
    stdout=server_log,
    stderr=subprocess.STDOUT
)

ready = False
for _ in range(30):
    try:
        with socket.create_connection(("127.0.0.1", 8000), timeout=1):
            ready = True
            break
    except OSError:
        time.sleep(1)

if not ready:
    server_log.flush()
    raise RuntimeError("MCP server failed to start. Check /content/nutrition_mcp_server.log")

MCP_SERVER_URL = "http://127.0.0.1:8000/mcp"
print("SUCCESS: MCP server is running:", MCP_SERVER_URL)
print("Server process ID:", server_process.pid)

SUCCESS: MCP server is running: http://127.0.0.1:8000/mcp
Server process ID: 3407


In [5]:
from mcp import Client

async def discover_nutrition_tools():
    async with Client(MCP_SERVER_URL) as mcp_client:
        result = await mcp_client.list_tools()
        print("SUCCESS: MCP client connected to Nutrition MCP Server")
        for tool in result.tools:
            print("\nTool:", tool.name)
            print("Description:", tool.description)
            print("Input schema:", tool.input_schema)
        return result.tools

discovered_tools = await discover_nutrition_tools()

SUCCESS: MCP client connected to Nutrition MCP Server

Tool: get_calories
Description: Return estimated calories (kcal) for a food and quantity in grams.
Input schema: {'type': 'object', 'properties': {'food': {'title': 'Food', 'type': 'string'}, 'quantity_g': {'default': 100.0, 'title': 'Quantity G', 'type': 'number'}}, 'required': ['food'], 'title': 'get_caloriesArguments'}

Tool: get_protein
Description: Return estimated protein in grams for a food and quantity in grams.
Input schema: {'type': 'object', 'properties': {'food': {'title': 'Food', 'type': 'string'}, 'quantity_g': {'default': 100.0, 'title': 'Quantity G', 'type': 'number'}}, 'required': ['food'], 'title': 'get_proteinArguments'}


In [6]:
import json

async def test_two_tools():
    async with Client(MCP_SERVER_URL) as mcp_client:
        calories = await mcp_client.call_tool(
            "get_calories", {"food": "chicken breast", "quantity_g": 100}
        )
        print("SUCCESSFUL TOOL CALL 1 — get_calories")
        print(json.dumps(calories.structured_content, indent=2))

        protein = await mcp_client.call_tool(
            "get_protein", {"food": "chicken breast", "quantity_g": 100}
        )
        print("\nSUCCESSFUL TOOL CALL 2 — get_protein")
        print(json.dumps(protein.structured_content, indent=2))

await test_two_tools()

SUCCESSFUL TOOL CALL 1 — get_calories
null

SUCCESSFUL TOOL CALL 2 — get_protein
null


In [7]:
import json

SYSTEM_PROMPT = """
You are NutriBuddy, a friendly, context-aware nutrition information assistant.
Use simple English.
For questions asking for calories or protein in a food, you MUST use the
matching MCP tool (get_calories or get_protein). Do not guess nutrition values.
Use the tool result in your answer. If the food is not in the dataset, explain that.
Do not diagnose diseases or prescribe medical treatment.
"""

# Keep existing session memory if you copied this into the earlier agent notebook.
if "conversation_history" not in globals():
    conversation_history = []


def convert_mcp_tools_to_groq(mcp_tools):
    return [{
        "type": "function",
        "function": {
            "name": tool.name,
            "description": tool.description or "",
            "parameters": tool.input_schema
        }
    } for tool in mcp_tools]


async def ask_nutrition_agent(user_message):
    global conversation_history
    conversation_history.append({"role": "user", "content": user_message})

    async with Client(MCP_SERVER_URL) as mcp_client:
        discovered = await mcp_client.list_tools()
        groq_tools = convert_mcp_tools_to_groq(discovered.tools)
        messages = [{"role": "system", "content": SYSTEM_PROMPT}]
        messages += conversation_history[-20:]

        for _ in range(5):
            response = groq_client.chat.completions.create(
                model=MODEL_NAME,
                messages=messages,
                tools=groq_tools,
                tool_choice="auto",
                temperature=0.2
            )
            assistant_message = response.choices[0].message

            if not assistant_message.tool_calls:
                answer = assistant_message.content or "I couldn't generate a response."
                conversation_history.append({"role": "assistant", "content": answer})
                return answer

            messages.append(assistant_message.model_dump(exclude_none=True))

            for tool_call in assistant_message.tool_calls:
                tool_name = tool_call.function.name
                arguments = json.loads(tool_call.function.arguments)
                print("\nAI selected MCP tool:", tool_name)
                print("Arguments:", arguments)

                result = await mcp_client.call_tool(tool_name, arguments)
                if result.structured_content is not None:
                    tool_output = json.dumps(result.structured_content)
                else:
                    tool_output = "\n".join(
                        block.text for block in result.content
                        if getattr(block, "type", None) == "text"
                    )
                print("MCP server returned:", tool_output)

                if result.is_error:
                    tool_output = "MCP tool error: " + tool_output

                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": tool_output
                })

    answer = "I couldn't complete the tool request."
    conversation_history.append({"role": "assistant", "content": answer})
    return answer

print("NutriBuddy is connected to the MCP tools!")

NutriBuddy is connected to the MCP tools!


In [8]:
answer = await ask_nutrition_agent(
    "How many calories are in 100g of chicken breast?"
)
print("\nNutriBuddy:", answer)


AI selected MCP tool: get_calories
Arguments: {'food': 'chicken breast', 'quantity_g': 100}
MCP server returned: {
  "food": "chicken breast",
  "quantity_g": 100.0,
  "calories_kcal": 165.0
}

NutriBuddy: 100 g of chicken breast contains about **165 kcal**.


In [9]:
answer = await ask_nutrition_agent(
    "How much protein is in 100g of chicken breast?"
)
print("\nNutriBuddy:", answer)


AI selected MCP tool: get_protein
Arguments: {'food': 'chicken breast', 'quantity_g': 100}
MCP server returned: {
  "food": "chicken breast",
  "quantity_g": 100.0,
  "protein_g": 31.0
}

NutriBuddy: 100 g of chicken breast provides roughly **31 g of protein**.
